In [2]:
import os 
import glob
import warnings

import numpy as np 
from utils.plots import *
from rl.agent import *
from utils.path import PathConfig

warnings.filterwarnings("ignore")
np.random.seed(0)

## Reinforcement learning (DRL) simulations 

This notebook performs RL simulations of the Pavlovian task from Tian & Uchida, 2015, using data-derived parameters:

- $\tau s $ derived from dopamine firing rates from Lhb lesion dataset
- $\eta $ derived from biophysical simulations (correspond to the asymmetric scaling factor given by receptor sensitivities, computed at a population level in **Figure 6i**)
- $\beta$ correspons to the decay factor in the updates for the $P$ and $N$ variables

Types of simulations  (`simulation_type`) are {`eta_sampling_`, `eta_nosampling_nodistr_`, `eta_nosampling_nodistr_`}


`eta_sampling_`: TD learning with DRL performing sampling for the computation of next step value predictors. 
$$
\delta_{i,t} = r_t + \gamma \cdot \hat{z}(s_{t+1}) - V_i(s_t)\\
\hat{\delta}_{i,t} ~~ = ~~~\tau_i \cdot \delta_{i,t} ~~~... ~~~\text{if} ~ \delta_{i,t} >0 \\
\hat{\delta}_{i,t}  = (1-\tau_i) \cdot \delta_{i,t} ~~~... ~~~\text{if} ~ \delta_{i,t} \leq 0
$$

- Where $ \hat{z}(s_{t+1})$ is a sample from the distribution defined by the set of $\hat{V}_i(s_{t+1})$
- Update of value:
$$
P_i(s_t) \leftarrow P_i(s_t) + \eta \cdot |\hat{\delta}_i(t)| - \beta \cdot P_i(s_t) ~~~... ~~~\text{if} ~ \delta_{i,t} >0\\
N_i(s_t) \leftarrow N_i(s_t) + (1-\eta) \cdot |\hat{\delta}_i(t)| - \beta \cdot N_i(s_t) ~~~... ~~~\text{if} ~ \delta_{i,t} \leq 0 \\
\hat{V}_i(s_t) = P_i(s_t) - N_i(s_t)
$$  

`eta_nosampling_nodistr_`: TD learning not DRL using the mean across  $\tau s $  
$$
\delta_{t} = r_t + \gamma \cdot V(s_{t+1}) - V(s_t)\\
\hat{\delta}_{t} ~~ = ~~~\hat{\tau} \cdot \delta_{t} ~~~... ~~~\text{if} ~ \delta_{t} >0 \\
\hat{\delta}_{t}  =(1-\hat{\tau}) \cdot \delta_{t} ~~~... ~~~\text{if} ~ \delta_{t} \leq 0
$$

- Where $ \hat{\tau}$ is the mean of the $\tau_i$ across the population of each group
- Update of value:
$$
P(s_t) \leftarrow P(s_t) + \eta \cdot |\hat{\delta}_i(t)| - \beta \cdot P(s_t) ~~~... ~~~\text{if} ~ \delta_{t} >0\\
N(s_t) \leftarrow N(s_t) + (1-\eta) \cdot |\hat{\delta}_i(t)| - \beta \cdot N(s_t) ~~~... ~~~\text{if} ~ \delta_{t} \leq 0 \\
\hat{V}(s_t) = P(s_t) - N(s_t)
$$  


`eta_nosampling_nodistr_`: TD learning with DRL without performing sampling for the computation of next step value predictors
$$
\delta_{i,t} = r_t + \gamma \cdot \hat{V}_i(s_{t+1}) - V_i(s_t)\\
\hat{\delta}_{i,t} ~~ = ~~~\tau_i \cdot \delta_{i,t} ~~~... ~~~\text{if} ~ \delta_{i,t} >0 \\
\hat{\delta}_{i,t}  = (1-\tau_i) \cdot \delta_{i,t} ~~~... ~~~\text{if} ~ \delta_{i,t} \leq 0
$$

- Where $ \hat{V}_i(s_{t+1})$  is the value predictor in the next time step
- Update of value:
$$
P_i(s_t) \leftarrow P_i(s_t) + \eta \cdot |\hat{\delta}_i(t)| - \beta \cdot P_i(s_t) ~~~... ~~~\text{if} ~ \delta_{i,t} >0\\
N_i(s_t) \leftarrow N_i(s_t) + (1-\eta) \cdot |\hat{\delta}_i(t)| - \beta \cdot N_i(s_t) ~~~... ~~~\text{if} ~ \delta_{i,t} \leq 0 \\
\hat{V}_i(s_t) = P_i(s_t) - N_i(s_t)
$$ 

**Note**: To plot and use the present notebook the user needs to download the data inside `analysis` folder of the OSF repository. Look at  `download_data` for the code to do this.

In [3]:
paths = PathConfig()
g_dir = paths.g_dir

save_dir = 'data/local/analysis/rl_simulations'
if not os.path.isdir(save_dir):
    os.makedirs(save_dir)


g_names = ['control','lesion']
if not os.path.isdir(save_dir):
    os.mkdir(save_dir)

taus_dic=np.load( os.path.join(g_dir,'analysis','taus_rec_occ_vs_da_level_population.npz'),allow_pickle=True )
taus_lin = taus_dic['taus_lin']
taus_lin[taus_lin>1] =np.nan

taus_dic_data=np.load(os.path.join(g_dir,'analysis','drl_metrics_from_data.npz'),allow_pickle=True )
taus_c = taus_dic_data['tau_control']
taus_l = taus_dic_data['tau_lesion']
taus_g = [taus_c,taus_l]




**Perform distributional reinforcement learning (DRL) simulations with :**
- $\tau s $ derived from dopamine firing rates from Lhb lesion dataset
- $\eta $ derived from biophysical simulations (correspond to the asymmetric scaling factor given by receptor sensitivities, computed at a population level in **Figure 6i**)
- $\beta$ correspons to the decay factor in the updates for the $P$ and $N$ variables

Types of simulations  (`simulation_type`) are {`eta_sampling_`, `eta_nosampling_nodistr_`, `eta_nosampling_nodistr_`}


`eta_sampling_`: TD learning with DRL performing sampling for the computation of next step value predictors. 
$$
\delta_{i,t} = r_t + \gamma \cdot \hat{z}(s_{t+1}) - V_i(s_t)\\
\hat{\delta}_{i,t} ~~ = ~~~\tau_i \cdot \delta_{i,t} ~~~... ~~~\text{if} ~ \delta_{i,t} >0 \\
\hat{\delta}_{i,t}  = (1-\tau_i) \cdot \delta_{i,t} ~~~... ~~~\text{if} ~ \delta_{i,t} \leq 0
$$

- Where $ \hat{z}(s_{t+1})$ is a sample from the distribution defined by the set of $\hat{V}_i(s_{t+1})$
- Update of value:
$$
P_i(s_t) \leftarrow P_i(s_t) + \eta \cdot |\hat{\delta}_i(t)| - \beta \cdot P_i(s_t) ~~~... ~~~\text{if} ~ \delta_{i,t} >0\\
N_i(s_t) \leftarrow N_i(s_t) + (1-\eta) \cdot |\hat{\delta}_i(t)| - \beta \cdot N_i(s_t) ~~~... ~~~\text{if} ~ \delta_{i,t} \leq 0 \\
\hat{V}_i(s_t) = P_i(s_t) - N_i(s_t)
$$  

`eta_nosampling_nodistr_`: TD learning not DRL using the mean across  $\tau s $  
$$
\delta_{t} = r_t + \gamma \cdot V(s_{t+1}) - V(s_t)\\
\hat{\delta}_{t} ~~ = ~~~\hat{\tau} \cdot \delta_{t} ~~~... ~~~\text{if} ~ \delta_{t} >0 \\
\hat{\delta}_{t}  =(1-\hat{\tau}) \cdot \delta_{t} ~~~... ~~~\text{if} ~ \delta_{t} \leq 0
$$

- Where $ \hat{\tau}$ is the mean of the $\tau_i$ across the population of each group
- Update of value:
$$
P(s_t) \leftarrow P(s_t) + \eta \cdot |\hat{\delta}_i(t)| - \beta \cdot P(s_t) ~~~... ~~~\text{if} ~ \delta_{t} >0\\
N(s_t) \leftarrow N(s_t) + (1-\eta) \cdot |\hat{\delta}_i(t)| - \beta \cdot N(s_t) ~~~... ~~~\text{if} ~ \delta_{t} \leq 0 \\
\hat{V}(s_t) = P(s_t) - N(s_t)
$$  


`eta_nosampling_nodistr_`: TD learning with DRL without performing sampling for the computation of next step value predictors
$$
\delta_{i,t} = r_t + \gamma \cdot \hat{V}_i(s_{t+1}) - V_i(s_t)\\
\hat{\delta}_{i,t} ~~ = ~~~\tau_i \cdot \delta_{i,t} ~~~... ~~~\text{if} ~ \delta_{i,t} >0 \\
\hat{\delta}_{i,t}  = (1-\tau_i) \cdot \delta_{i,t} ~~~... ~~~\text{if} ~ \delta_{i,t} \leq 0
$$

- Where $ \hat{V}_i(s_{t+1})$  is the value predictor in the next time step
- Update of value:
$$
P_i(s_t) \leftarrow P_i(s_t) + \eta \cdot |\hat{\delta}_i(t)| - \beta \cdot P_i(s_t) ~~~... ~~~\text{if} ~ \delta_{i,t} >0\\
N_i(s_t) \leftarrow N_i(s_t) + (1-\eta) \cdot |\hat{\delta}_i(t)| - \beta \cdot N_i(s_t) ~~~... ~~~\text{if} ~ \delta_{i,t} \leq 0 \\
\hat{V}_i(s_t) = P_i(s_t) - N_i(s_t)
$$ 


In [13]:
simulation_type = 'eta_sampling_' 
beta_v = [0.001,0.002,0.005,0.01,0.015,0.02] 
for ieps in np.arange(len(beta_v)):
    epsilon_ = beta_v[ieps]
    for i_group in  np.arange(2):
        results_v = list()
        etas =taus_lin[i_group,:]        
        n_iterations = 25 
        taus_= np.sort(taus_g[i_group])
        taus_ = taus_[(taus_>0)*(taus_<1)]
        #% %
        for ie in np.arange(n_iterations):
            if not  np.isnan(etas[ie] ):
                s = dict()
                s['agent'] = dict()
                s['task'] = dict()
                if simulation_type == 'eta_nosampling_' :
                    s['agent']['taus'] =  np.sort(taus_) # needs to be array  np.asarray([np.nanmean(taus_) ]) #  np.sort(taus_)
                    s['agent']['do_sampling'] = False
                elif simulation_type == 'eta_nosampling_nodistr_': 
                    s['agent']['taus'] =   np.asarray([np.nanmean(taus_) ]) # needs to be array 
                    s['agent']['do_sampling'] = False
                elif simulation_type == 'eta_sampling_': 
                    s['agent']['taus'] =  np.sort(taus_) # needs to be array  np.asarray([np.nanmean(taus_) ]) #  np.sort(taus_)
                    s['agent']['do_sampling'] = True

                s['agent']['eta'] = etas[ie]
                print('Processing for eta = '+ str(etas[ie]) + 'epsilon = '+str(epsilon_) + '|| ' +str(ie) + ' of '+str(n_iterations))
                s['agent']['str_agent'] = ['Eta_' + str(etas[ie])]#,'neutral','optimistic']
                s['agent']['epsilon'] = epsilon_
                s['agent']['base_alpha'] = 1 
                s['agent']['gamma'] = 0.99
                
                s['agent']['d1_d2'] = True
                s['task']['prob'] = [[.1, .9],[.5, .5],[.9, .1] ,[.8, .2]]
                s['task']['mags'] = [[1, 0],[1, 0],[1, 0] ,[-1, 0]]
                s['task']['percent_per_cs'] = 1/len(s['task']['mags'])*np.ones((len(s['task']['mags']),1))
                s['task']['cs_id'] = [[0,0],[1,1],[2,2],[3,3]]
                s['task']['str_cs'] = [ '10%','50%','90%','80%Puff']
                s['task']['str_task'] = 'drl_6odor'
                s['task']['n_trials'] = 2000
                s['task']['cue_onset'] = 1
                s['task']['cue_dur'] = 1
                s['task']['rew_onset'] = 3
                s['task']['iti_dur'] = 1
                s['task']['n_iterations'] = 1
                s['task']['type'] = 'cue_reward'

                agent = TDLearning(s)
                agent.initialize_results()
                agent.run_task()
                agent.reformat_results()
                save_dic = dict()
                results = dict()
                save_vars = ['distribution_trials_full', 'distribution_g_trials_full', 'distribution_ng_trials_full',  'deltas_trials_full', 
                             'distribution_ng_trials', 'distribution_g_trials', 'distribution_trials','deltas_trials']
                for ivar in save_vars:
                    temp = agent.results[ivar]
                    if len(temp.shape)==4:
                        id_end = np.argwhere(np.nansum(temp,axis=(0,1,2))==0).flatten()[0]
                        temp = temp[:,:,:,:id_end]
                    results[ivar] = temp
                save_vars = ['distribution','distribution_g','distribution_ng','samp_distribution','deltas','deltas_trials_raw','distribution_std']
                new_dict = vars(agent)
                for ivar in save_vars:
                    results[ivar] = new_dict[ivar]
                agent_dict = dict()
                save_vars=['taus', 'eta', 'str_agent', 'epsilon', 'base_alpha', 'do_sampling', 'gamma',
                            'n_cells', 'task_type', 'n_deliv_rew', 'deliv_rew', 'deliv_cs', 'trial_length', 
                            'n_trials', 'cs_vector', 'us_vector', 'id_states_per_cs', 'n_types', 'n_tr_max', 
                            'i_step_per_cs']
                for ivar in save_vars:
                    agent_dict[ivar] = new_dict[ivar]
                save_dic['results'] = results
                save_dic['agent'] = agent_dict
                save_dic['task'] = vars(agent.task)
                save_dic['eta'] = etas
                np.save(save_dir+'results_sim_type_'+ simulation_type 
                + 'eps_' + str(epsilon_).replace('.','_')+'_'+g_names[i_group]+'_iteration_'+str(ie) ,save_dic)
                print('Saving ' + save_dir+'results_sim_type_'+simulation_type 
                + 'eps_' + str(epsilon_).replace('.','_')+'_'+g_names[i_group]+'_iteration_'+str(ie) )

Processing for eta = 0.5485640794749749epsilon = 0.001|| 0 of 25
Trial 0 of 2000  ||| 
Trial 1000 of 2000  ||| 


KeyboardInterrupt: 